# Install + imports

In [ ]:
!pip -q install timm

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF

import timm
from tqdm.auto import tqdm


import gc
import json
from pathlib import Path

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/thanhkhil/efficientnet-custom")
from efficientnet_custom import (
    efficientnet_b2_original,
    efficientnet_b2_gem,
    efficientnet_b2_late_eca,
    efficientnet_b2_gem_late_eca,
    EfficientNetB2Custom
)

In [ ]:
import inspect
print(inspect.getsource(EfficientNetB2Custom.reset_classifier))

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/thanhkhil/efficientnet-dualscale")
from efficientnet_dualscale_1 import (
    efficientnet_b2_dual_scale,
    # efficientnet_b2_dual_scale_gem,
    # efficientnet_b2_dual_scale_eca,
    # efficientnet_b2_dual_scale_eca_gem,
    # efficientnet_b2_dual_scale_arcface,
    # efficientnet_b2_dual_scale_gated,
    # efficientnet_b2_dual_scale_aux
)

# Global config

In [ ]:
seed = 123 #42, 123, 2024

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_name = "skin31"

skin31_root = Path("/kaggle/input/datasets/kelixo25/31-classes-of-skin-disease/Atlas dan ISIC2019 (31 classes)")
skin31_train_dir = skin31_root / "train"
skin31_test_dir = skin31_root / "test"

ham_root = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
ham_metadata_path = ham_root / "HAM10000_metadata.csv"
ham_part1_dir = ham_root / "HAM10000_images_part_1"
ham_part2_dir = ham_root / "HAM10000_images_part_2"

work_dir = Path("/kaggle/working")
history_dir = work_dir / "histories"
history_dir.mkdir(parents=True, exist_ok=True)
class_names_path = work_dir / f"class_names_{dataset_name}.json"

img_size = 224
batch_size = 32
epochs = 15
lr = 5e-4
weight_decay = 1e-6
num_workers = 0

# Dataset overview

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

HAM_DX_TO_NAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions"
}

def build_ham10000_dataframe(metadata_path, part1_dir, part2_dir):
    df = pd.read_csv(metadata_path).copy()

    def resolve_path(image_id):
        p1 = part1_dir / f"{image_id}.jpg"
        p2 = part2_dir / f"{image_id}.jpg"
        if p1.exists():
            return str(p1)
        if p2.exists():
            return str(p2)
        return None

    df["path"] = df["image_id"].map(resolve_path)
    df = df[df["path"].notna()].reset_index(drop=True)

    dx_order = sorted(df["dx"].unique().tolist())
    class_names = [HAM_DX_TO_NAME.get(x, x) for x in dx_order]
    dx_to_idx = {dx: i for i, dx in enumerate(dx_order)}

    df["target"] = df["dx"].map(dx_to_idx)
    df["class_name"] = df["dx"].map(lambda x: HAM_DX_TO_NAME.get(x, x))

    return df, dx_order, class_names, dx_to_idx

class HAM10000Dataset(Dataset):
    def __init__(self, df, class_names, transform=None, exclude_paths=None):
        self.df = df.copy().reset_index(drop=True)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        if len(exclude_paths) > 0:
            self.df = self.df[~self.df["path"].isin(exclude_paths)].reset_index(drop=True)

        self.transform = transform
        self.classes = class_names
        self.samples = list(zip(self.df["path"].tolist(), self.df["target"].tolist()))
        self.imgs = self.samples
        self.targets = self.df["target"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        target = int(row["target"])

        if self.transform is not None:
            image = self.transform(image)

        return image, target


def build_ham10000_split(test_size=0.2, seed=42):
    ham_df, ham_dx_order, ham_class_names, ham_dx_to_idx = build_ham10000_dataframe(
        metadata_path=ham_metadata_path,
        part1_dir=ham_part1_dir,
        part2_dir=ham_part2_dir
    )

    lesion_df = (
        ham_df.groupby("lesion_id", as_index=False)["target"]
        .first()
        .copy()
    )

    train_lesions, test_lesions = train_test_split(
        lesion_df["lesion_id"],
        test_size=test_size,
        random_state=seed,
        stratify=lesion_df["target"]
    )

    train_df = ham_df[ham_df["lesion_id"].isin(train_lesions)].reset_index(drop=True)
    test_df = ham_df[ham_df["lesion_id"].isin(test_lesions)].reset_index(drop=True)

    train_df = train_df.copy()
    test_df = test_df.copy()

    return train_df, test_df, ham_class_names

In [ ]:
def summarize_ham_split(train_df, test_df, class_names):
    train_counts = train_df["target"].value_counts().sort_index()
    test_counts = test_df["target"].value_counts().sort_index()

    summary_df = pd.DataFrame({
        "class_name": class_names,
        "train_count": [int(train_counts.get(i, 0)) for i in range(len(class_names))],
        "test_count": [int(test_counts.get(i, 0)) for i in range(len(class_names))]
    })

    print("Train images:", len(train_df))
    print("Test images:", len(test_df))
    print("Train lesions:", train_df["lesion_id"].nunique())
    print("Test lesions:", test_df["lesion_id"].nunique())

    display(summary_df)

In [ ]:
ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=42)
summarize_ham_split(ham_train_df, ham_test_df, class_names)

In [ ]:
if dataset_name == "skin31":
    base_train = datasets.ImageFolder(skin31_train_dir)
    base_test = datasets.ImageFolder(skin31_test_dir)

    class_names = base_train.classes
    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

elif dataset_name == "ham10000":
    ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=seed)

    base_train = HAM10000Dataset(ham_train_df, class_names=class_names, transform=None)
    base_test = HAM10000Dataset(ham_test_df, class_names=class_names, transform=None)

    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

df_counts = pd.DataFrame({
    "class_name": class_names,
    "train_count": train_counts.values,
    "test_count": test_counts.values
})

class_meta = {
    "dataset_name": dataset_name,
    "num_classes": num_classes,
    "class_names": class_names
}

with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_meta, f, ensure_ascii=False, indent=2)

print("Dataset:", dataset_name)
print("Num classes:", num_classes)
print("Train samples:", len(base_train))
print("Test samples:", len(base_test))

display(df_counts)

# Custom transforms

In [ ]:
class FixedCenterZoom:
    def __init__(self, scale=1.2):
        self.scale = scale

    def __call__(self, img):
        w, h = img.size
        nw, nh = int(w * self.scale), int(h * self.scale)
        img = img.resize((nw, nh), Image.BILINEAR)
        left = (nw - w) // 2
        top = (nh - h) // 2
        return img.crop((left, top, left + w, top + h))


class FixedRotate:
    def __init__(self, angle=20):
        self.angle = angle

    def __call__(self, img):
        return TF.rotate(img, self.angle, interpolation=transforms.InterpolationMode.BILINEAR, fill=0)


class FixedBrightness:
    def __init__(self, factor=1.2):
        self.factor = factor

    def __call__(self, img):
        return ImageEnhance.Brightness(img).enhance(self.factor)


class FixedShear:
    def __init__(self, shear=12):
        self.shear = shear

    def __call__(self, img):
        return TF.affine(
            img,
            angle=0,
            translate=[0, 0],
            scale=1.0,
            shear=[self.shear, 0],
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0
        )

sample_path, sample_label = base_train.samples[0]
sample_img = Image.open(sample_path).convert("RGB")

preview = {
    "Original": sample_img,
    "Center Zoom": FixedCenterZoom(1.2)(sample_img.copy()),
    "Rotation": FixedRotate(20)(sample_img.copy()),
    "Brightness": FixedBrightness(1.2)(sample_img.copy()),
    "Shear": FixedShear(12)(sample_img.copy()),
    "Vertical Flip": TF.vflip(sample_img.copy()),
    "Horizontal Flip": TF.hflip(sample_img.copy())
}

fig, axes = plt.subplots(1, 7, figsize=(24, 4))

for ax, (name, img) in zip(axes, preview.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Sample class:", class_names[sample_label])

# Dataset filtering helper

In [ ]:
class FilteredImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None, exclude_paths=None):
        super().__init__(root=root, transform=transform)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        filtered_samples = []
        filtered_targets = []

        for path, target in self.samples:
            if str(Path(path)) not in exclude_paths:
                filtered_samples.append((path, target))
                filtered_targets.append(target)

        self.samples = filtered_samples
        self.imgs = filtered_samples
        self.targets = filtered_targets

# Transform definitions

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

to_tensor_norm = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])
orig_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

zoom_tf = transforms.Compose([
    FixedCenterZoom(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

rot_tf = transforms.Compose([
    FixedRotate(20),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

bright_tf = transforms.Compose([
    FixedBrightness(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

shear_tf = transforms.Compose([
    FixedShear(12),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

vflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.vflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

hflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.hflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Training utilities

In [ ]:
def rand_bbox(size, lam):
    H = size[2]
    W = size[3]

    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    return x1, y1, x2, y2


def apply_cutmix(images, labels, alpha=1.0):
    if alpha <= 0:
        return images, labels, labels, 1.0

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)

    labels_a = labels
    labels_b = labels[rand_index]

    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images = images.clone()
    images[:, :, y1:y2, x1:x2] = images[rand_index, :, y1:y2, x1:x2]

    lam = 1.0 - ((x2 - x1) * (y2 - y1) / (images.size(2) * images.size(3)))

    return images, labels_a, labels_b, lam

In [ ]:
class SAM(Optimizer):
    def __init__(self, params, base_optimizer_cls, rho=0.05, adaptive=False, **kwargs):
        if rho < 0.0:
            raise ValueError(f"Invalid rho, should be non-negative: {rho}")

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super().__init__(params, defaults)

        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)
        self.rho = rho
        self.adaptive = adaptive

    @torch.no_grad()
    def _grad_norm(self):
        device = self.param_groups[0]["params"][0].device
        norms = []

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                if group.get("adaptive", self.adaptive):
                    norms.append((torch.abs(p) * p.grad).norm(p=2).to(device))
                else:
                    norms.append(p.grad.norm(p=2).to(device))

        if len(norms) == 0:
            return torch.tensor(0.0, device=device)

        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()

        if grad_norm.item() == 0.0:
            if zero_grad:
                self.zero_grad(set_to_none=True)
            return

        for group in self.param_groups:
            scale = group.get("rho", self.rho) / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None:
                    continue

                if group.get("adaptive", self.adaptive):
                    e_w = torch.pow(p, 2) * p.grad * scale
                else:
                    e_w = p.grad * scale

                p.add_(e_w)
                self.state[p]["e_w"] = e_w

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = self.state[p].get("e_w", None)
                if e_w is not None:
                    p.sub_(e_w)

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def step(self, closure=None):
        raise NotImplementedError("Use first_step and second_step with SAM.")

    def zero_grad(self, set_to_none=True):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

In [ ]:
def prepare_batch(images, labels, use_cutmix=False, cutmix_alpha=1.0):
    if use_cutmix:
        images, labels_a, labels_b, lam = apply_cutmix(images, labels, alpha=cutmix_alpha)
        return images, labels_a, labels_b, lam
    return images, labels, labels, 1.0


def forward_loss(
    model,
    images,
    labels,
    criterion,
    labels_a=None,
    labels_b=None,
    lam=1.0,
    use_amp=True,
    aux_weight=0.2
):
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    uses_margin_head = getattr(model, "uses_margin_head", False)
    uses_aux_head = getattr(model, "uses_aux_head", False)

    with torch.amp.autocast(amp_device, enabled=use_amp and torch.cuda.is_available()):
        if uses_margin_head:
            if labels_a is not None and labels_b is not None and lam != 1.0:
                outputs_a = model(images, target=labels_a)
                outputs_b = model(images, target=labels_b)
                loss = lam * criterion(outputs_a, labels_a) + (1.0 - lam) * criterion(outputs_b, labels_b)
                outputs = lam * outputs_a + (1.0 - lam) * outputs_b
            else:
                outputs = model(images, target=labels)
                loss = criterion(outputs, labels)

        elif uses_aux_head:
            main_outputs, aux_outputs = model(images, return_aux=True)

            if labels_a is not None and labels_b is not None and lam != 1.0:
                main_loss = lam * criterion(main_outputs, labels_a) + (1.0 - lam) * criterion(main_outputs, labels_b)
                aux_loss = lam * criterion(aux_outputs, labels_a) + (1.0 - lam) * criterion(aux_outputs, labels_b)
            else:
                main_loss = criterion(main_outputs, labels)
                aux_loss = criterion(aux_outputs, labels)

            loss = main_loss + aux_weight * aux_loss
            outputs = main_outputs

        else:
            outputs = model(images)
            if labels_a is not None and labels_b is not None and lam != 1.0:
                loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                loss = criterion(outputs, labels)

    return outputs, loss


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    aux_weight=0.2
):
    model.train()

    running_loss = 0.0
    correct = 0.0
    total = 0

    pbar = tqdm(loader, leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = prepare_batch(
            images=images,
            labels=labels,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha
        )

        optimizer.zero_grad(set_to_none=True)

        if not use_sam:
            outputs, loss = forward_loss(
                model=model,
                images=images,
                labels=labels,
                criterion=criterion,
                labels_a=labels_a if use_cutmix else None,
                labels_b=labels_b if use_cutmix else None,
                lam=lam,
                use_amp=True,
                aux_weight=aux_weight
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss detected: {loss.item()}")

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            if use_cutmix:
                correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
            else:
                correct += (preds == labels).sum().item()

            total += labels.size(0)
            pbar.set_postfix(loss=running_loss / total, acc=correct / total)
            continue

        outputs_1, loss_1 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight
        )

        if not torch.isfinite(loss_1):
            raise RuntimeError(f"Non-finite loss detected at SAM first pass: {loss_1.item()}")

        loss_1.backward()
        optimizer.first_step(zero_grad=True)

        _, loss_2 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight
        )

        if not torch.isfinite(loss_2):
            raise RuntimeError(f"Non-finite loss detected at SAM second pass: {loss_2.item()}")

        loss_2.backward()
        optimizer.second_step(zero_grad=False)
        optimizer.base_optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss_1.item() * images.size(0)
        preds = outputs_1.argmax(dim=1)

        if use_cutmix:
            correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
        else:
            correct += (preds == labels).sum().item()

        total += labels.size(0)
        pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"

    with torch.no_grad():
        pbar = tqdm(loader, leave=False)

        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(amp_device, enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite eval loss detected: {loss.item()}")

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

# Model/optimizer/scheduler utilities

In [ ]:
def build_model(
    model_name="efficientnet_b2",
    model_variant="original",
    drop_rate=0.0,
    pretrained=True,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=30.0,
    arcface_m=0.3
):
    if model_name != "efficientnet_b2":
        model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            drop_rate=drop_rate
        )
    else:
        if model_variant == "original":
            model = efficientnet_b2_original(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "gem":
            model = efficientnet_b2_gem(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "late_eca":
            model = efficientnet_b2_late_eca(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "gem_late_eca":
            model = efficientnet_b2_gem_late_eca(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "dual_scale":
            model = efficientnet_b2_dual_scale(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        elif model_variant == "dual_scale_eca":
            model = efficientnet_b2_dual_scale_eca(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        elif model_variant == "dual_scale_eca_gem":
            model = efficientnet_b2_dual_scale_eca_gem(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels,
                gem_p=3.0,
                gem_eps=1e-6
            )
        elif model_variant == "dual_scale_gem":
            model = efficientnet_b2_dual_scale_gem(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "dual_scale_arcface":
            model = efficientnet_b2_dual_scale_arcface(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels,
                embedding_dim=embedding_dim,
                arcface_s=arcface_s,
                arcface_m=arcface_m
            )
        elif model_variant == "dual_scale_gated":
            model = efficientnet_b2_dual_scale_gated(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        elif model_variant == "dual_scale_aux":
            model = efficientnet_b2_dual_scale_aux(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        else:
            raise ValueError(f"Unsupported model_variant: {model_variant}")

    return model.to(device)


def build_optimizer(model, optimizer_name, lr_value=None, use_sam=False, sam_rho=0.05, sam_adaptive=False):
    actual_lr = lr if lr_value is None else lr_value

    if not use_sam:
        if optimizer_name == "adam":
            return torch.optim.Adam(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        if optimizer_name == "adamw":
            return torch.optim.AdamW(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    if optimizer_name == "adam":
        return SAM(
            model.parameters(),
            torch.optim.Adam,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    if optimizer_name == "adamw":
        return SAM(
            model.parameters(),
            torch.optim.AdamW,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    raise ValueError(f"Unsupported optimizer: {optimizer_name}")


def build_scheduler(optimizer, scheduler_name):
    target_optimizer = optimizer.base_optimizer if hasattr(optimizer, "base_optimizer") else optimizer

    if scheduler_name == "none":
        return None
    if scheduler_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(target_optimizer, T_max=epochs)
    raise ValueError(f"Unsupported scheduler: {scheduler_name}")


def slugify_exp_name(exp_name):
    return exp_name.replace(" ", "_").replace("+", "plus").lower()


def save_history_files(exp_name, history):
    exp_slug = slugify_exp_name(exp_name)
    hist_df = pd.DataFrame(history)
    csv_path = history_dir / f"{exp_slug}_history.csv"
    json_path = history_dir / f"{exp_slug}_history.json"
    hist_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    return csv_path, json_path


def save_summary_files(all_results):
    summary_df = pd.DataFrame(all_results)
    csv_path = work_dir / "experiment_summary.csv"
    json_path = work_dir / "experiment_summary.json"
    summary_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    return summary_df, csv_path, json_path


def run_experiment(
    exp_name,
    optimizer_name,
    scheduler_name,
    model_name="efficientnet_b2",
    model_variant="original",
    lr_value=None,
    label_smoothing=0.0,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    sam_rho=0.05,
    sam_adaptive=False,
    drop_rate=0.0,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=30.0,
    arcface_m=0.3,
    aux_weight=0.2
):
    actual_lr = lr if lr_value is None else lr_value

    print("=" * 80)
    print(f"Running: {exp_name} | model={model_name} | variant={model_variant} | lr={actual_lr} | sam={use_sam}")
    print("=" * 80)

    exp_slug = slugify_exp_name(exp_name)
    save_path = work_dir / f"{exp_slug}_best.pth"

    model = build_model(
        model_name=model_name,
        model_variant=model_variant,
        drop_rate=drop_rate,
        pretrained=True,
        fusion_channels=fusion_channels,
        embedding_dim=embedding_dim,
        arcface_s=arcface_s,
        arcface_m=arcface_m
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = build_optimizer(
        model=model,
        optimizer_name=optimizer_name,
        lr_value=lr_value,
        use_sam=use_sam,
        sam_rho=sam_rho,
        sam_adaptive=sam_adaptive
    )
    scheduler = build_scheduler(optimizer, scheduler_name)

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=torch.cuda.is_available())

    best_acc = 0.0
    best_epoch = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha,
            use_sam=use_sam,
            aux_weight=aux_weight
        )

        val_loss, val_acc, y_true, y_pred = evaluate(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=device
        )

        if scheduler is not None:
            scheduler.step()

        row = {
            "experiment": exp_name,
            "model_name": model_name,
            "model_variant": model_variant,
            "optimizer": optimizer_name,
            "scheduler": scheduler_name,
            "lr": float(actual_lr),
            "label_smoothing": float(label_smoothing),
            "use_cutmix": bool(use_cutmix),
            "cutmix_alpha": float(cutmix_alpha),
            "use_sam": bool(use_sam),
            "sam_rho": float(sam_rho),
            "sam_adaptive": bool(sam_adaptive),
            "drop_rate": float(drop_rate),
            "fusion_channels": int(fusion_channels),
            "embedding_dim": int(embedding_dim),
            "arcface_s": float(arcface_s),
            "arcface_m": float(arcface_m),
            "aux_weight": float(aux_weight),
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "test_loss": float(val_loss),
            "test_acc": float(val_acc),
        }
        history.append(row)

        save_history_files(exp_name, history)

        print(
            f"[{exp_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"lr={actual_lr:.6f} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"test_loss={val_loss:.4f} | "
            f"test_acc={val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = float(val_acc)
            best_epoch = epoch
            torch.save(model.state_dict(), save_path)

    print(f"[{exp_name}] Best test accuracy: {best_acc:.4f} at epoch {best_epoch}")
    print(f"[{exp_name}] Saved best model to: {save_path}")

    del model, criterion, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "experiment": exp_name,
        "model_name": model_name,
        "model_variant": model_variant,
        "optimizer": optimizer_name,
        "scheduler": scheduler_name,
        "lr": float(actual_lr),
        "label_smoothing": float(label_smoothing),
        "use_cutmix": bool(use_cutmix),
        "cutmix_alpha": float(cutmix_alpha),
        "use_sam": bool(use_sam),
        "sam_rho": float(sam_rho),
        "sam_adaptive": bool(sam_adaptive),
        "drop_rate": float(drop_rate),
        "fusion_channels": int(fusion_channels),
        "embedding_dim": int(embedding_dim),
        "arcface_s": float(arcface_s),
        "arcface_m": float(arcface_m),
        "aux_weight": float(aux_weight),
        "best_acc": best_acc,
        "best_epoch": best_epoch,
        "save_path": str(save_path),
        "status": "ok"
    }

In [ ]:
model = efficientnet_b2_dual_scale(num_classes=num_classes, pretrained=False).to(device)

x = torch.randn(2, 3, 260, 260).to(device)

with torch.no_grad():
    c4, c5 = model.extract_dual_features(x)

print("C4:", c4.shape)
print("C5:", c5.shape)

In [ ]:
model = efficientnet_b2_dual_scale(
    num_classes=num_classes,
    pretrained=False,
    drop_rate=0.0,
    fusion_channels=512
).to(device)

x = torch.randn(2, 3, 260, 260).to(device)
out = model(x)
print("logits:", out.shape)

shape_info = model.debug_shapes(torch.randn(2, 3, 260, 260).to(device))
print(shape_info)

# Analysis utilities

In [ ]:
def collect_predictions_with_metadata(model, loader, dataset, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []
    all_confidences = []
    all_paths = []

    sample_paths = [path for path, _ in dataset.samples]

    offset = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device, non_blocking=True)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            confidences, preds = probs.max(dim=1)

            batch_size_now = labels.size(0)

            all_labels.extend(labels.numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_confidences.extend(confidences.cpu().numpy().tolist())
            all_paths.extend(sample_paths[offset:offset + batch_size_now])

            offset += batch_size_now

    df = pd.DataFrame({
        "path": all_paths,
        "true_idx": all_labels,
        "pred_idx": all_preds,
        "confidence": all_confidences
    })

    df["true_class"] = df["true_idx"].map(lambda x: class_names[x])
    df["pred_class"] = df["pred_idx"].map(lambda x: class_names[x])
    df["correct"] = df["true_idx"] == df["pred_idx"]

    return df, np.array(all_probs)

In [ ]:
def show_samples(df_samples, n=9):
    df_show = df_samples.head(n).copy()
    cols = 3
    rows = int(np.ceil(len(df_show) / cols))

    plt.figure(figsize=(14, 4 * rows))

    for i, (_, row) in enumerate(df_show.iterrows(), 1):
        img = Image.open(row["path"]).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(
            f'True: {row["true_class"]}\nPred: {row["pred_class"]}\nConf: {row["confidence"]:.4f}'
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
path = "/kaggle/input/models/thanhkhil/efficientskindis/tensorflow2/default/1/Weights.h5"

model = models.efficientnet_b2(weights=None).to(device)

in_features = 1024
model._fc = nn.Sequential(
    nn.BatchNorm1d(num_features=in_features),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.BatchNorm1d(num_features=128),
    nn.Dropout(0.4),
    nn.Linear(128, 31)
).to(device)

ckpt = torch.load(path, map_location="cpu")
missing, unexpected = model.load_state_dict(ckpt, strict=False)

print("Missing:", len(missing))
print("Unexpected:", len(unexpected))
print(missing[:20])
print(unexpected[:20])

In [ ]:
x = torch.randn(1, 3, 224, 224).to(device)
model.eval()
with torch.no_grad():
    y = model(x)
print(y.shape)

# Load best checkpoint for diagnosis

In [ ]:
best_exp_name = "Baseline + LS0.1 + CutMix1.0 + SAM0.1 + lr2.5e-4"
best_model_path = Path("/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/baseline-ls-cutmix-sam/1/baseline_plus_ls0.1_plus_cutmix1.0_plus_sam0.1_plus_lr2.5e-4_best.pth")

print(best_model_path)
print(best_model_path.exists())

In [ ]:
def build_test_dataset_with_paths():
    if dataset_name == "skin31":
        return datasets.ImageFolder(skin31_test_dir, transform=to_tensor_norm)

    if dataset_name == "ham10000":
        return HAM10000Dataset(
            ham_test_df,
            class_names=class_names,
            transform=to_tensor_norm
        )

    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

analysis_test_dataset = build_test_dataset_with_paths()
analysis_test_loader = DataLoader(
    analysis_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

In [ ]:
if dataset_name == "skin31":
    teacher_analysis_model = build_model(
        model_name="efficientnet_b2",
        drop_rate=0.0,
        pretrained=False
    )
    teacher_analysis_model.load_state_dict(torch.load(best_model_path, map_location=device))
    teacher_analysis_model.eval()

    analysis_test_dataset = build_test_dataset_with_paths()
    analysis_test_loader = DataLoader(
        analysis_test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    df_pred, prob_matrix = collect_predictions_with_metadata(
        teacher_analysis_model,
        analysis_test_loader,
        analysis_test_dataset,
        device
    )

# Analyze train set

In [ ]:
# train_analysis_dataset = datasets.ImageFolder(train_dir, transform=to_tensor_norm)

# train_analysis_loader = DataLoader(
#     train_analysis_dataset,
#     batch_size=batch_size,
#     shuffle=False,
#     num_workers=num_workers,
#     pin_memory=True
# )

# df_train_pred, train_prob_matrix = collect_predictions_with_metadata(
#     model=teacher_analysis_model,
#     loader=train_analysis_loader,
#     dataset=train_analysis_dataset,
#     device=device
# )

# print(df_train_pred.head())
# print(df_train_pred["correct"].mean())

In [ ]:
# train_report = classification_report(
#     df_train_pred["true_idx"],
#     df_train_pred["pred_idx"],
#     target_names=class_names,
#     output_dict=True,
#     zero_division=0
# )

# df_train_report = pd.DataFrame(train_report).transpose()
# df_train_report

# df_train_class_report = df_train_report.iloc[:-3].copy()
# df_train_class_report = df_train_class_report.sort_values(by="f1-score", ascending=True)
# df_train_class_report[["precision", "recall", "f1-score", "support"]].head(10)

In [ ]:
# train_cm = confusion_matrix(df_train_pred["true_idx"], df_train_pred["pred_idx"])

# plt.figure(figsize=(14, 12))
# plt.imshow(train_cm, interpolation="nearest")
# plt.colorbar()
# plt.xticks(range(num_classes), class_names, rotation=90)
# plt.yticks(range(num_classes), class_names)
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.title("Train Confusion Matrix")
# plt.tight_layout()
# plt.show()

In [ ]:
# train_cm_row_sum = train_cm.sum(axis=1, keepdims=True)
# train_cm_norm = np.divide(train_cm, train_cm_row_sum, where=train_cm_row_sum != 0)

# plt.figure(figsize=(14, 12))
# plt.imshow(train_cm_norm, interpolation="nearest")
# plt.colorbar()
# plt.xticks(range(num_classes), class_names, rotation=90)
# plt.yticks(range(num_classes), class_names)
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.title("Train Normalized Confusion Matrix")
# plt.tight_layout()
# plt.show()

In [ ]:
# df_train_wrong = df_train_pred[df_train_pred["correct"] == False].copy()
# df_train_wrong = df_train_wrong.sort_values(by="confidence", ascending=False)
# df_train_wrong.head(20)

In [ ]:
# show_samples(df_train_wrong, n=9)

In [ ]:
# df_train_correct = df_train_pred[df_train_pred["correct"] == True].copy()
# df_train_correct = df_train_correct.sort_values(by="confidence", ascending=True)
# df_train_correct.head(20)

In [ ]:
# train_cm_no_diag = train_cm.copy()
# np.fill_diagonal(train_cm_no_diag, 0)

# train_pairs = []
# for i in range(num_classes):
#     for j in range(num_classes):
#         if train_cm_no_diag[i, j] > 0:
#             train_pairs.append({
#                 "true_class": class_names[i],
#                 "pred_class": class_names[j],
#                 "count": int(train_cm_no_diag[i, j])
#             })

# df_train_pairs = pd.DataFrame(train_pairs).sort_values(by="count", ascending=False)
# df_train_pairs.head(20)

In [ ]:
# train_analysis_dir = work_dir / "train_analysis_outputs"
# train_analysis_dir.mkdir(parents=True, exist_ok=True)

# df_train_pred.to_csv(train_analysis_dir / "train_predictions.csv", index=False)
# df_train_wrong.to_csv(train_analysis_dir / "train_hard_wrong_samples.csv", index=False)
# df_train_correct.to_csv(train_analysis_dir / "train_low_conf_correct_samples.csv", index=False)
# df_train_class_report.to_csv(train_analysis_dir / "train_per_class_report.csv")
# df_train_pairs.to_csv(train_analysis_dir / "train_top_confusion_pairs.csv", index=False)

# print("Saved train analysis files to:", train_analysis_dir)

In [ ]:
# import shutil
# # Nén thư mục 'analysis_outputs' thành file 'outputs.zip'
# shutil.make_archive('train_analysis_outputs', 'zip', '/kaggle/working/train_analysis_outputs')

In [ ]:
# candidate_df = df_train_wrong[
#     (df_train_wrong["true_class"] == "Nevus") &
#     (df_train_wrong["pred_class"] == "Actinic keratosis")
# ].copy()

# candidate_df = candidate_df.sort_values(by="confidence", ascending=False)
# candidate_df.head(20)

In [ ]:
# show_samples(candidate_df, n=20)

# Refined training dataset

In [ ]:
exclude_paths = []

if dataset_name == "skin31":
    train_sets = [
        FilteredImageFolder(skin31_train_dir, transform=orig_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=zoom_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=rot_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=bright_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=shear_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=vflip_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=hflip_tf, exclude_paths=exclude_paths)
    ]

    train_dataset = ConcatDataset(train_sets)
    test_dataset = datasets.ImageFolder(skin31_test_dir, transform=to_tensor_norm)

    base_train_refined = FilteredImageFolder(skin31_train_dir, transform=None, exclude_paths=exclude_paths)
    base_targets = np.array(base_train_refined.targets)

elif dataset_name == "ham10000":
    train_sets = [
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=orig_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=zoom_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=rot_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=bright_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=shear_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=vflip_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=hflip_tf, exclude_paths=exclude_paths)
    ]

    train_dataset = ConcatDataset(train_sets)
    test_dataset = HAM10000Dataset(ham_test_df, class_names=class_names, transform=to_tensor_norm)

    base_train_refined = HAM10000Dataset(ham_train_df, class_names=class_names, transform=None, exclude_paths=exclude_paths)
    base_targets = np.array(base_train_refined.targets)

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

class_counts = np.bincount(base_targets, minlength=num_classes)
class_weights = np.zeros_like(class_counts, dtype=np.float64)

nonzero_mask = class_counts > 0
class_weights[nonzero_mask] = 1.0 / class_counts[nonzero_mask]
sample_weights = class_weights[base_targets]
sample_weights = np.tile(sample_weights, len(train_sets))

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print("Dataset:", dataset_name)
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))
print("Num classes:", num_classes)

if "teacher_analysis_model" in globals():
    del teacher_analysis_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Retrain experiments

In [ ]:

experiments = [    
    # {
    #     "exp_name": "Adam + cosine",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine"
    # },
    # {
    #     "exp_name": "Adam + no scheduler",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "none"
    # },
    # {
    #     "exp_name": "AdamW + no scheduler",
    #     "optimizer_name": "adamw",
    #     "scheduler_name": "none"
    # },
    # {
    #     "exp_name": "AdamW + cosine",
    #     "optimizer_name": "adamw",
    #     "scheduler_name": "cosine"
    # },
    # {
    #     "exp_name": "Adam + cosine + LS0.1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + CutMix1.0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.0,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.03",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.03
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.05",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.05
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.1 + lr2.5e-4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.2,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.3",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.3,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.4,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Baseline + LS0.1 + CutMix1.0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0
    # },
    # {
    #     "exp_name": "Baseline + LS0.1 + CutMix1.0 + SAM0.1 + lr2.5e-4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "B0 baseline",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": False,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "B0 + SAM0.1",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "B0 + SAM0.1 + lr2.5e-4",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "gem",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + late ECA",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "late_eca",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    {
        "exp_name": "Strong B2 + Dual Scale",
        "model_name": "efficientnet_b2",
        "model_variant": "dual_scale",
        "fusion_channels": 512,
        "optimizer_name": "adam",
        "scheduler_name": "cosine",
        "label_smoothing": 0.1,
        "use_cutmix": True,
        "cutmix_alpha": 1.0,
        "lr_value": 2.5e-4,
        "use_sam": True,
        "sam_rho": 0.1
    },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ECA",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_eca",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,        
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ECA + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_eca_gem",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_gem",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ArcFace",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_arcface",
    #     "fusion_channels": 512,
    #     "embedding_dim": 256,
    #     "arcface_s": 30.0,
    #     "arcface_m": 0.3,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + Gated Fusion",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_gated",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + Aux C4",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_aux",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "aux_weight": 0.2
    # },
    # {
    #     "exp_name": "HAM10000 Strong Baseline",
    #     "model_name": "efficientnet_b2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "HAM10000 Strong Baseline + Dual Scale",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # }
]

In [ ]:
all_results = []

for exp in experiments:
    try:
        result = run_experiment(
            exp_name=exp["exp_name"],
            optimizer_name=exp["optimizer_name"],
            scheduler_name=exp["scheduler_name"],
            lr_value=exp.get("lr_value", None),
            model_name=exp.get("model_name", "efficientnet_b2"),
            model_variant=exp.get("model_variant", "original"),
            label_smoothing=exp.get("label_smoothing", 0.0),
            use_cutmix=exp.get("use_cutmix", False),
            cutmix_alpha=exp.get("cutmix_alpha", 1.0),
            use_sam=exp.get("use_sam", False),
            sam_rho=exp.get("sam_rho", 0.05),
            sam_adaptive=exp.get("sam_adaptive", False),
            drop_rate=exp.get("drop_rate", 0.0),
            fusion_channels=exp.get("fusion_channels", 512),
            embedding_dim=exp.get("embedding_dim", 256),
            arcface_s=exp.get("arcface_s", 30.0),
            arcface_m=exp.get("arcface_m", 0.3),
            aux_weight=exp.get("aux_weight", 0.2)
        )
    except Exception as e:
        result = {
            "experiment": exp["exp_name"],
            "model_name": exp.get("model_name", "efficientnet_b2"),
            "model_variant": exp.get("model_variant", "original"),
            "optimizer": exp["optimizer_name"],
            "scheduler": exp["scheduler_name"],
            "lr": float(exp.get("lr_value", lr)),
            "label_smoothing": float(exp.get("label_smoothing", 0.0)),
            "use_cutmix": bool(exp.get("use_cutmix", False)),
            "cutmix_alpha": float(exp.get("cutmix_alpha", 1.0)),
            "use_sam": bool(exp.get("use_sam", False)),
            "sam_rho": float(exp.get("sam_rho", 0.05)),
            "sam_adaptive": bool(exp.get("sam_adaptive", False)),
            "drop_rate": float(exp.get("drop_rate", 0.0)),
            "fusion_channels": int(exp.get("fusion_channels", 512)),
            "embedding_dim": int(exp.get("embedding_dim", 256)),
            "arcface_s": float(exp.get("arcface_s", 30.0)),
            "arcface_m": float(exp.get("arcface_m", 0.3)),
            "aux_weight": float(exp.get("aux_weight", 0.2)),
            "best_acc": None,
            "best_epoch": None,
            "save_path": None,
            "status": f"failed: {repr(e)}"
        }
        print(f"[{exp['exp_name']}] FAILED -> {repr(e)}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    all_results.append(result)
    summary_df, summary_csv_path, summary_json_path = save_summary_files(all_results)

summary_df = pd.DataFrame(all_results).sort_values(
    by=["best_acc"],
    ascending=False,
    na_position="last"
).reset_index(drop=True)

summary_df.to_csv(work_dir / "experiment_summary_sorted.csv", index=False)

print("\nFinal summary:")
display(summary_df)

all_history_list = []
for exp in experiments:
    exp_slug = slugify_exp_name(exp["exp_name"])
    csv_path = history_dir / f"{exp_slug}_history.csv"
    if csv_path.exists():
        df_exp = pd.read_csv(csv_path)
        all_history_list.append(df_exp)

if len(all_history_list) > 0:
    all_history = pd.concat(all_history_list, ignore_index=True)
    all_history.to_csv(work_dir / "all_history.csv", index=False)
    display(all_history)

    plt.figure(figsize=(12, 5))
    for exp_name in all_history["experiment"].unique():
        df_exp = all_history[all_history["experiment"] == exp_name]
        plt.plot(df_exp["epoch"], df_exp["test_acc"], label=exp_name)
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("test_acc")
    plt.title("Test Accuracy")
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name in all_history["experiment"].unique():
        df_exp = all_history[all_history["experiment"] == exp_name]
        plt.plot(df_exp["epoch"], df_exp["train_acc"], label=exp_name)
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("train_acc")
    plt.title("Train Accuracy")
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name in all_history["experiment"].unique():
        df_exp = all_history[all_history["experiment"] == exp_name]
        plt.plot(df_exp["epoch"], df_exp["test_loss"], label=exp_name)
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("test_loss")
    plt.title("Test Loss")
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name in all_history["experiment"].unique():
        df_exp = all_history[all_history["experiment"] == exp_name]
        plt.plot(df_exp["epoch"], df_exp["train_loss"], label=exp_name)
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("train_loss")
    plt.title("Train Loss")
    plt.show()
else:
    print("No history files found.")